In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import MNIST
from tqdm.notebook import tqdm

### Prep Dataset ###
tensor_transforms = transforms.Compose(
    [
        transforms.Resize((32,32)),
        transforms.ToTensor()
    ]
)

train_set = MNIST("../../../data/mnist/", train=True, transform=tensor_transforms)
test_set = MNIST("../../../data/mnist/", train=False, transform=tensor_transforms)

### Set Device ###
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
class VectorQuantizer(nn.Module):

    def __init__(self, codebook_size=1024, latent_dim=2):
        super().__init__()
        self.embedding = nn.Embedding(codebook_size, latent_dim)
        self.embedding.weight.data.uniform_(-1/codebook_size, 1/codebook_size)
        self.latent_dim = latent_dim
        self.codebook_size = codebook_size


    def forward(self, x):
        batch_size = x.shape[0]
        # distance between every latent and code: (L-C)**2 = (L**2 - 2LC + C**2)

        L2 = torch.sum(x**2, dim=1, keepdim=True)
        C2 = torch.sum(self.embedding.weight**2, dim=1).unsqueeze(0)
        CL = x @ self.embedding.weight.t()
        distances = L2 - 2*CL + C2
        closest = torch.argmin(distances, dim=-1)

        # create empty quantized latents embedding
        quantized_latents_idx = torch.zeros(batch_size, self.codebook_size, device=x.device)

        # place 1 at the indexes for each sample for the codebook we want
        batch_idx = torch.arange(batch_size)
        quantized_latents_idx[batch_idx, closest] = 1

        # matrix multiplication to grab indexed latents from embeddings
        quantized_latents = quantized_latents_idx @ self.embedding.weight

        return quantized_latents, closest

In [ ]:
class LinearResidualVectorQuantizer(nn.Module):

    def __init__(self, latent_dim=2, codebook_size=64, num_codebooks=4):
        super().__init__()

        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Linear(32*32, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, latent_dim)
        )


        self.rvq = nn.ModuleList(
            [
                VectorQuantizer(codebook_size, latent_dim) for _ in range(num_codebooks)
            ]
        )


        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.Linear(),
            nn.Linear(128, 32*32),
            nn.Sigmoid()
        )


    def quantize(self, z, return_code_idx_only=False):

        codebook_loss = 0
        commitment_loss = 0

        quantized_codes = []
        code_indexes = []

        final_quantized = torch.zeros_like(z)

        for quantizer in self.rvq:

            codes, code_idx = quantizer(z)
            code_indexes.append(code_idx)

            codebook_loss = torch.mean((codes - z.detach())**2)
            commitment_loss = torch.mean((codes.detach() - z)**2)

            codebook_loss += codebook_loss
            commitment_loss += commitment_loss

            codes = z + (codes - z).detach()

            final_quantized = final_quantized + codes

            quantized_codes.append(codes)

            z = z - codes.detach()


        if return_code_idx_only:
            return torch.stack(code_indexes, dim=-1)

        else:
            return final_quantized, codebook_loss, commitment_loss


    def forward_enc(self, x):
        return self.encoder(x)


    def forward_dec(self, x):
        final_quantized, codebook_loss, commitment_loss = self.quantize(x)
        decoded = self.decoder(final_quantized)
        return final_quantized, decoded, codebook_loss, commitment_loss


    def forward(self, x):
        batch_size, channels, height, width = x.shape
        x = x.flatten(1)

        latents = self.forward_enc(x)
        final_quantized, decoded, codebook_loss, commitment_loss = self.forward_dec(latents)

        decoded = decoded.reshape(batch_size, channels, height, width)
        return latents, final_quantized, decoded, codebook_loss, commitment_loss